# Tugas 4 | Ekstraksi -> TFIDF & Word2Vec

Ekstrkasi dilakukan menggunakan *TF-IDF* dan *Word Embedding* untuk mendapatkan perbandingan mana yang lebih optimal.

## TF-IDF

In [5]:
# Import library
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Baca dataset
df = pd.read_csv("detik_cleaned.csv")

# 2. Ambil kolom teks (pakai clean_stemmed biar sudah bersih)
texts = df["clean_stemmed"].dropna()

# 3. TF-IDF Vectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

# 4. Konversi ke DataFrame
tfidf_df = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out()
)

# (opsional) Tambahkan kolom kategori biar bisa dipakai untuk klasifikasi
tfidf_df.insert(0, "Kategori", df.loc[texts.index, "Kategori"].values)

# 5. Simpan ke CSV
tfidf_df.to_csv("tfidf_result.csv", index=False)

print("TF-IDF berhasil disimpan sebagai tfidf_result.csv")


TF-IDF berhasil disimpan sebagai tfidf_result.csv


## Word Embedding

In [3]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec

# 1. Load dataset
df = pd.read_csv("detik_cleaned.csv")

# 2. Ambil teks hasil stemming (sudah bersih)
texts = df["clean_stemmed"].dropna()

# 3. Tokenisasi
tokenized_texts = [t.split() for t in texts]

# 4. Latih Word2Vec
w2v_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,   # dimensi embedding
    window=5,
    min_count=1,
    workers=4,
    sg=1
)

# 5. Representasi dokumen = rata-rata embedding kata
doc_embeddings = []
for tokens in tokenized_texts:
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    if vectors:
        doc_embeddings.append(np.mean(vectors, axis=0))
    else:
        doc_embeddings.append(np.zeros(100))  # kalau dokumen kosong

doc_embeddings = np.array(doc_embeddings)

# 6. Konversi ke DataFrame
embedding_df = pd.DataFrame(doc_embeddings)

# Opsional: tambahkan kategori biar bisa dipakai untuk analisis/klasifikasi
embedding_df.insert(0, "Kategori", df.loc[texts.index, "Kategori"].values)

# 7. Simpan ke CSV
embedding_df.to_csv("word2vec_result.csv", index=False)

print("Word Embedding berhasil disimpan sebagai word2vec_result.csv")


Word Embedding berhasil disimpan sebagai word2vec_result.csv


## Dimensi Keduanya

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
import numpy as np

# 1. Load dataset
df = pd.read_csv("detik_cleaned.csv")
texts = df["clean_stemmed"].dropna()
tokenized = [t.split() for t in texts]  # tokenisasi sederhana

# 2. TF-IDF
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(texts)
print("Dimensi TF-IDF:", X_tfidf.shape[1])

# 3. Word2Vec
w2v_model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=4) #Melatih model Word2Vec dengan vektor berdimensi 100.

# Buat representasi dokumen dengan rata-rata embedding kata
doc_embeddings = []
for tokens in tokenized:
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    if vectors:
        doc_embeddings.append(np.mean(vectors, axis=0))
    else:
        doc_embeddings.append(np.zeros(100))
doc_embeddings = np.array(doc_embeddings)

print("Dimensi Word2Vec:", doc_embeddings.shape[1])


Dimensi TF-IDF: 3439
Dimensi Word2Vec: 100


stemming/stopword, ubah ubah paramater embedding sampai menemukan yang bagus